In [ ]:
folder = '.'

In [ ]:
# load
import json
import pathlib
import pandas as pd

folder = pathlib.Path(folder)
assert folder.exists()

# aggregate results into csv necessary
file_list = list(folder.glob('out*.json'))
dict_list = list()
for file in file_list:
    with open(file, 'r') as f:
        dict_list.append(json.load(f))
    
# load aggregated results
f_csv = folder / 'results.csv'
if f_csv.exists():
    df = pd.read_csv(f_csv, index_col=None)
else:
    df = pd.DataFrame()

# add in existing result
df = pd.concat((df, pd.DataFrame(dict_list)))

# round p_value to 14 decimal places (avoids floating point comparison failure)
df['p_val'] = df['p_val'].round(14)

# drop duplicates & check for conflicting results
df.drop_duplicates(inplace=True)
assert df.value_counts(subset=['p_val', 'seed', 'Analysis']).max() == 1
    
# overwrite csv with latest / greatest
df.to_csv(f_csv, index=False)

# delete json files (they're in csv)
for file in file_list:
    file.unlink()

In [ ]:
import numpy as np
from collections import defaultdict


# extract
pval_list = sorted(df['p_val'].unique())
seed_list = sorted(df['seed'].unique())

shape = len(seed_list), len(pval_list)
score_dict = defaultdict(lambda: np.full(shape=shape, fill_value=np.nan))

for _, row in df.iterrows():
    seed_idx = seed_list.index(row['seed'])
    pval_idx = pval_list.index(row['p_val'])
    
    for feat in ('f1', 'sens', 'spec'):
        score_dict[row['Analysis'], feat][seed_idx, pval_idx] = row[feat]

In [ ]:
# plot
import seaborn as sns
import matplotlib.pyplot as plt

sns.set()

fig, ax = plt.subplots(2, 3)

# plot top row
style_dict = {'AnalysisTFCE': {'color': 'r'}, 
              'AnalysisHGLM': {'color': 'b'}}
style_single = {'linewidth': .5,
                'zorder': 1,
                'label': '_nolegend_'}
style_mean = {'linewidth': 5,
              'zorder': 2,
              'label': '_nolegend_'}
for _ax, feat in zip(ax[0, :], ('f1', 'sens', 'spec')):
    plt.sca(_ax)
    for method, kwargs in style_dict.items():
        plt.plot(pval_list, score_dict[method, feat].T, **kwargs, **style_single)
        plt.plot(pval_list, np.nanmean(score_dict[method, feat], axis=0), **kwargs, **style_mean)

    plt.xlabel('p_val')
    plt.ylabel(feat)
    plt.xscale('log')

# plot bottom row
for _ax, feat in zip(ax[1, :], ('f1', 'sens', 'spec')):
    plt.sca(_ax)
    x = score_dict['AnalysisHGLM', feat] - score_dict['AnalysisTFCE', feat]
    plt.axhline([0], linewidth=2, color='k')
    plt.plot(pval_list, x.T, color='k', **style_single)
    plt.plot(pval_list, np.nanmean(x, axis=0), color='k', **style_mean)

    plt.xlabel('p_val')
    plt.ylabel(f'{feat}: hglm - TFCE')
    plt.xscale('log')
    
# add legend in last plot of top row
plt.sca(ax[0, -1])
del style_single['label']
for method, kwargs in style_dict.items():
    plt.plot([], [], label=method[-4:], **kwargs, **style_single)
plt.legend()
    
fig.set_size_inches(10, 6)
fig.tight_layout()
fig.savefig(folder / 'hglm_vs_TFCE.png', bbox_inches='tight')

# Get worst case scenario



In [ ]:
import cloudpickle as pickle
import gzip

def get_uuid(match_dict):
    """ gets series of all uuid values matching the input dict """
    s_bool = pd.Series(True, index=df.index)
    for col, val in match_dict.items():
        s_bool &= df[col] == val
    return df[s_bool]['uuid']

def load(uuid):
    """ loads detail file """
    file_list = list(folder.glob(f'*{uuid}_detail*'))
    assert len(file_list) == 1
    file = file_list[0]
    
    with gzip.open(file, 'rb') as f:
        x = pickle.load(f)

    return x

In [ ]:
f1_diff = score_dict['AnalysisHGLM', 'f1'] - score_dict['AnalysisTFCE', 'f1']
idx = np.where(f1_diff == np.nanmin(f1_diff))
pval_min = pval_list[idx[1][0]]
seed_min = seed_list[idx[0][0]]

s_uuid = get_uuid({'p_val': pval_min, 'seed': seed_min, 'Analysis': 'AnalysisHGLM'})
ana, effect = load(s_uuid.iloc[0])

In [ ]:
from plotly

def scatter_size_vs_stat_plotly(epoch, y_feat, mask=None, min_size=1, y_feat_label='y_feat'):
    """ scatters size vs f_stat, colors by f1 score if mask is passed

     Args:
        epoch (Epoch):
        y_feat (np.array): y feature to plot (same size as size)
        mask (np.array): target mask
        min_size (int): smallest size to be plotted
     """
    size = epoch.size.astype(float)
    n_perm, n_reg = size.shape
    reg_idx, perm_idx = np.meshgrid(range(n_perm), range(n_reg))
    df = pd.DataFrame({'size': size.flatten(),
                  y_feat_label: y_feat.flatten(),
                  'perm_idx': perm_idx.flatten(),
                  'reg_idx': reg_idx, flatten()})
    
    # compute f1 score
    if mask is not None:
        df['f1'] = get_f1(mask=mask,
                    mask_idx=epoch.exp.mask_idx,
                    children=epoch.child_dict[0]).flatten()

    assert isinstance(y_feat, np.ndarray)
    assert y_feat.shape == size.shape
    y = copy(y_feat)

    if min_size > 1:
        df.drop(df['size'] < min_size, inplace=True)
    
    if mask is None:
        plt.scatter(size, y, alpha=.02, color='k',
                    linewidth=0, label='region')
    else:
        plt.scatter(size[1:, :], y[1:, :], alpha=.02, color='k',
                    linewidth=0, label='region (permuted)')

        b = f1 > 0
        plt.scatter(size[0, :][b], y[0, :][b], c=f1.flatten()[b],
                    cmap='plasma', label='region w/ target', marker='s')
        cbar = plt.colorbar()
        cbar.set_label('F1 score', rotation=90)

    plt.xlabel('size')
    plt.xscale('log')
    plt.yscale('log')
    plt.legend()

In [ ]:
size = np.ones((3, 5))
n_perm, n_reg = size.shape
permute_idx = np.tile(np.atleast_2d(np.arange(n_perm)).T, (1, n_reg))
permute_idx = np.tile(np.atleast_2d(np.arange(n_reg)), (n_perm, ))

In [ ]:
reg_idx, perm_idx = np.meshgrid(range(n_perm), range(n_reg))
reg_idx

In [ ]:
scatter_size_vs_stat_plotly(ana.epoch_list[0], ana.epoch_list[0].z_stat, mask=effect.mask)